# Colección y Descripción de Datos — Internet Fijo Accesos por Tecnología y Segmento

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, isnan, regexp_replace, mean, sum as Fsum

In [0]:
spark = SparkSession.builder.appName("MinEducacion_Proyecto").getOrCreate()

file_path = "/Volumes/workspace/proyecto/proyectvolume/dataframes/dataframes/Internet-Fijo-Accesos-por-Tecnología-y-Segmento.csv"
df_internet = spark.read.csv(file_path, header=True, inferSchema=False, sep=",")

display(df_internet.limit(15))
# para verlos todos se qutia el limit
#display(df_internet)

AÑO,TRIMESTRE,PROVEEDOR,COD_DEPARTAMENTO,DEPARTAMENTO,COD_MUNICIPIO,MUNICIPIO,SEGMENTO,TECNOLOGIA,VELOCIDAD_BAJADA,VELOCIDAD_SUBIDA,No DE ACCESOS
2019,3,COMUNICACION CELULAR S A COMCEL S A,15,BOYACÁ,15238,DUITAMA,CORPORATIVO,CABLE,"300,00","10,00",5
2020,4,COMUNICACION CELULAR S A COMCEL S A,15,BOYACÁ,15759,SOGAMOSO,RESIDENCIAL - ESTRATO 3,CABLE,"15,00","1,02",44
2020,4,L@TINNETWORK SAS,15,BOYACÁ,15299,GARAGOA,RESIDENCIAL - ESTRATO 2,FIBER TO THE HOME (FTTH),"8,00","8,00",4
2020,1,ACCESS DIGITAL S.A.S,15,BOYACÁ,15835,TURMEQUÉ,CORPORATIVO,WIFI,"4,00","1,20",1
2017,4,FUTURE SOLUTIONS DEVELOPMENT S.A.S.,15,BOYACÁ,15542,PESCA,RESIDENCIAL - ESTRATO 2,OTRAS TECNOLOGÍAS INALÁMBRICAS,"4,00","4,00",6
2018,4,DIRECTV COLOMBIA LTDA,15,BOYACÁ,15491,NOBSA,RESIDENCIAL - ESTRATO 3,OTRAS TECNOLOGÍAS INALÁMBRICAS,"6,00","1,00",10
2020,1,WI-SAT COMUNICACIONES S.A.S.,15,BOYACÁ,15238,DUITAMA,CORPORATIVO,OTRAS TECNOLOGÍAS INALÁMBRICAS,"5,00","2,00",5
2020,4,COLOMBIA TELECOMUNICACIONES S.A. E.S.P.,15,BOYACÁ,15176,CHIQUINQUIRÁ,CORPORATIVO,XDSL,"3,00","1,00",11
2019,3,COLOMBIA TELECOMUNICACIONES S.A. E.S.P.,15,BOYACÁ,15516,PAIPA,CORPORATIVO,XDSL,"1,00","1,00",4
2019,1,FUTURE SOLUTIONS DEVELOPMENT S.A.S.,15,BOYACÁ,15759,SOGAMOSO,CORPORATIVO,OTRAS TECNOLOGÍAS INALÁMBRICAS,"8,00","8,00",1


In [0]:
print("--- ESQUEMA DE DATOS ---")
df_internet.printSchema()

print(f"\nTotal de registros (filas): {df_internet.count()}")
print(f"Total de atributos (columnas): {len(df_internet.columns)}")

--- ESQUEMA DE DATOS ---
root
 |-- AÑO: string (nullable = true)
 |-- TRIMESTRE: string (nullable = true)
 |-- PROVEEDOR: string (nullable = true)
 |-- COD_DEPARTAMENTO: string (nullable = true)
 |-- DEPARTAMENTO: string (nullable = true)
 |-- COD_MUNICIPIO: string (nullable = true)
 |-- MUNICIPIO: string (nullable = true)
 |-- SEGMENTO: string (nullable = true)
 |-- TECNOLOGIA: string (nullable = true)
 |-- VELOCIDAD_BAJADA: string (nullable = true)
 |-- VELOCIDAD_SUBIDA: string (nullable = true)
 |-- No DE ACCESOS: string (nullable = true)


Total de registros (filas): 143370
Total de atributos (columnas): 12


## Comprensión del significado de atributos clave:

**AÑO / TRIMESTRE**: Período de reporte del dato (2016–2023, trimestres 1 a 4).

**PROVEEDOR**: Empresa prestadora del servicio de internet fijo.

**COD_DEPARTAMENTO / DEPARTAMENTO**: Código y nombre del departamento (15 = Boyacá).

**COD_MUNICIPIO / MUNICIPIO**: Código DANE y nombre del municipio donde se presta el servicio.

**SEGMENTO**: Clasificación del suscriptor: residencial (estratos 1–6), corporativo, sin estratificar o uso propio del operador.

**TECNOLOGIA**: Tecnología usada para la conexión (FTTH, CABLE, XDSL, SATELITAL, WIFI, entre otras).

**VELOCIDAD_BAJADA / VELOCIDAD_SUBIDA**: Velocidad contratada en Mbps para descarga y carga. Vienen con coma decimal en lugar de punto.

**No DE ACCESOS**: Número de suscriptores activos para esa combinación de proveedor, municipio, tecnología y segmento en ese período.

#Exploración de los Datos
Se presentan 8 elementos exploratorios para comprender la distribución
de la conectividad fija en los municipios de Boyacá entre 2016 y 2023.

In [0]:
df_exp = df_internet \
    .withColumn("VELOCIDAD_BAJADA_NUM", regexp_replace(col("VELOCIDAD_BAJADA"), ",", ".").cast("float")) \
    .withColumn("VELOCIDAD_SUBIDA_NUM", regexp_replace(col("VELOCIDAD_SUBIDA"), ",", ".").cast("float")) \
    .withColumn("NUM_ACCESOS", col("`No DE ACCESOS`").cast("integer"))

#1: Estadísticos descriptivos de velocidades y accesos
print("Elemento 1: Estadísticos de Velocidad de Bajada (Mbps)")
df_exp.select("VELOCIDAD_BAJADA_NUM").summary().show()

print("Elemento 2: Estadísticos de Velocidad de Subida (Mbps)")
df_exp.select("VELOCIDAD_SUBIDA_NUM").summary().show()

#3: Registros por año
print("Elemento 3: Cantidad de registros por Año")
df_exp.groupBy("AÑO").count().orderBy("AÑO").show()

#4: Total de accesos por año
print("Elemento 4: Total de accesos (suscriptores) por Año")
df_exp.groupBy("AÑO") \
    .agg(Fsum("NUM_ACCESOS").alias("total_accesos")) \
    .orderBy("AÑO").show()

Elemento 1: Estadísticos de Velocidad de Bajada (Mbps)
+-------+--------------------+
|summary|VELOCIDAD_BAJADA_NUM|
+-------+--------------------+
|  count|              143370|
|   mean|   73.81507930295716|
| stddev|   769.2706250713508|
|    min|                 0.0|
|    25%|                 5.0|
|    50%|                10.0|
|    75%|                40.0|
|    max|            100000.0|
+-------+--------------------+

Elemento 2: Estadísticos de Velocidad de Subida (Mbps)
+-------+--------------------+
|summary|VELOCIDAD_SUBIDA_NUM|
+-------+--------------------+
|  count|              143370|
|   mean|   40.43523910100869|
| stddev|  500.70285684717476|
|    min|                 0.0|
|    25%|                 2.0|
|    50%|                 5.0|
|    75%|                12.0|
|    max|             71680.0|
+-------+--------------------+

Elemento 3: Cantidad de registros por Año
+----+-----+
| AÑO|count|
+----+-----+
|2016|   49|
|2017| 8801|
|2018|13159|
|2019|16902|
|2020|20335

In [0]:
#5: Evolución de accesos por año
display(
    df_exp.groupBy("AÑO")
    .agg(Fsum("NUM_ACCESOS").alias("total_accesos"))
    .orderBy("AÑO")
)

#6: Accesos por segmento
display(
    df_exp.groupBy("SEGMENTO")
    .agg(Fsum("NUM_ACCESOS").alias("total_accesos"))
    .orderBy("total_accesos", ascending=False)
)

#7: Accesos por tecnología
display(
    df_exp.groupBy("TECNOLOGIA")
    .agg(Fsum("NUM_ACCESOS").alias("total_accesos"))
    .orderBy("total_accesos", ascending=False)
)

#8: Top 10 municipios por accesos
display(
    df_exp.groupBy("MUNICIPIO")
    .agg(Fsum("NUM_ACCESOS").alias("total_accesos"))
    .orderBy("total_accesos", ascending=False)
    .limit(10)
)

AÑO,total_accesos
2016,414
2017,348267
2018,451308
2019,484846
2020,520002
2021,616331
2022,632086
2023,484378


SEGMENTO,total_accesos
RESIDENCIAL - ESTRATO 2,1654710
RESIDENCIAL - ESTRATO 3,816468
RESIDENCIAL - ESTRATO 1,423507
CORPORATIVO,360062
RESIDENCIAL - ESTRATO 4,206076
SIN ESTRATIFICAR,40010
RESIDENCIAL - ESTRATO 5,34991
RESIDENCIAL - ESTRATO 6,1391
USO PROPIO INTERNO DEL OPERADOR,417


TECNOLOGIA,total_accesos
CABLE,1344822
XDSL,708605
FIBER TO THE HOME (FTTH),623228
HYBRID FIBER COAXIAL (HFC),360011
OTRAS TECNOLOGÍAS INALÁMBRICAS,141559
FIBER TO THE CABINET (FTTC),118224
OTRAS TECNOLOGÍAS DE FIBRA (ANTES FTTX),114074
WIFI,95076
SATELITAL,16711
OTRAS TECNOLOGÍAS FIJAS,8399


MUNICIPIO,total_accesos
TUNJA,1184976
SOGAMOSO,689715
DUITAMA,678843
PUERTO BOYACÁ,155058
CHIQUINQUIRÁ,141889
PAIPA,138349
VILLA DE LEYVA,62434
NOBSA,35274
MONIQUIRÁ,27388
TIBASOSA,25582


#Reporte de Calidad de Datos

In [0]:
from pyspark.sql.functions import col, count, when, isnan

print("--- REPORTE DE VALORES FALTANTES ---")

expresiones_nulos = []
for columna, tipo in df_internet.dtypes:
    if tipo == 'string':
        condicion = col(columna).isNull() | (col(columna) == "") | (col(columna) == "NaN")
    else:
        condicion = col(columna).isNull() | isnan(columna)
    expresiones_nulos.append(count(when(condicion, columna)).alias(columna))

df_internet.select(expresiones_nulos).show(vertical=True)

--- REPORTE DE VALORES FALTANTES ---
-RECORD 0---------------
 AÑO              | 0   
 TRIMESTRE        | 0   
 PROVEEDOR        | 0   
 COD_DEPARTAMENTO | 0   
 DEPARTAMENTO     | 0   
 COD_MUNICIPIO    | 0   
 MUNICIPIO        | 0   
 SEGMENTO         | 0   
 TECNOLOGIA       | 0   
 VELOCIDAD_BAJADA | 0   
 VELOCIDAD_SUBIDA | 0   
 No DE ACCESOS    | 0   



In [0]:
df_exp.select(
    count(when(col("VELOCIDAD_BAJADA_NUM") == 0, 1)).alias("velocidad_bajada_cero"),
    count(when(col("VELOCIDAD_SUBIDA_NUM") == 0, 1)).alias("velocidad_subida_cero"),
    count(when(col("NUM_ACCESOS") == 0, 1)).alias("accesos_cero")
).show()

print("Registros por segmento a eliminar:")
df_internet.filter(
    col("SEGMENTO").isin("USO PROPIO INTERNO DEL OPERADOR", "SIN ESTRATIFICAR")
).groupBy("SEGMENTO").count().show(truncate=False)

+---------------------+---------------------+------------+
|velocidad_bajada_cero|velocidad_subida_cero|accesos_cero|
+---------------------+---------------------+------------+
|                  256|                  262|        1648|
+---------------------+---------------------+------------+

Registros por segmento a eliminar:
+-------------------------------+-----+
|SEGMENTO                       |count|
+-------------------------------+-----+
|SIN ESTRATIFICAR               |1560 |
|USO PROPIO INTERNO DEL OPERADOR|161  |
+-------------------------------+-----+



## Técnicas propuestas para tratar valores problemáticos:

#### Eliminación de registros con accesos == 0:
Representan combinaciones de proveedor/municipio/tecnología sin suscriptores reales.
No aportan información al análisis de cobertura y son eliminados con un filtro.

#### Eliminación del segmento "USO PROPIO INTERNO DEL OPERADOR":
Solo 161 registros (0.1%) que corresponden a uso interno de los operadores,
no representan acceso real.

#### Eliminación de columnas redundantes:
COD_DEPARTAMENTO y DEPARTAMENTo se eliminan porque el dataset ya está
filtrado a Boyacá.

#### Velocidades en 0: 
Se dejan como nulos en lugar de cero para promedios en análisis futuros.

#Filtros, Limpieza y Transformación Inicial

In [0]:
print(f"Filas antes de filtros: {df_internet.count()}")

df_internet = df_internet \
    .withColumn("VELOCIDAD_BAJADA", regexp_replace(col("VELOCIDAD_BAJADA"), ",", ".").cast("float")) \
    .withColumn("VELOCIDAD_SUBIDA", regexp_replace(col("VELOCIDAD_SUBIDA"), ",", ".").cast("float")) \
    .withColumn("No DE ACCESOS", col("`No DE ACCESOS`").cast("integer"))

#1: Eliminar accesos == 0
df_internet = df_internet.filter(col("No DE ACCESOS") > 0)
print(f"Filas tras eliminar accesos == 0: {df_internet.count()}")

#2: Eliminar segmento
df_internet = df_internet.filter(col("SEGMENTO") != "USO PROPIO INTERNO DEL OPERADOR")
print(f"Filas tras eliminar uso propio operador: {df_internet.count()}")

Filas antes de filtros: 143370
Filas tras eliminar accesos == 0: 141722
Filas tras eliminar uso propio operador: 141561


In [0]:
#1: Eliminar columnas redundantes
df_internet = df_internet.drop("COD_DEPARTAMENTO", "DEPARTAMENTO")
print(f"Columnas tras eliminación: {len(df_internet.columns)}")

#2: 0 = null
df_internet = df_internet \
    .withColumn("VELOCIDAD_BAJADA", when(col("VELOCIDAD_BAJADA") == 0, None).otherwise(col("VELOCIDAD_BAJADA"))) \
    .withColumn("VELOCIDAD_SUBIDA", when(col("VELOCIDAD_SUBIDA") == 0, None).otherwise(col("VELOCIDAD_SUBIDA")))

# 3: Creacion d columna
df_internet = df_internet.withColumn(
    "TIPO_TECNOLOGIA",
    when(col("TECNOLOGIA").contains("FIBER") | col("TECNOLOGIA").contains("FTTX"), "FIBRA")
    .when(col("TECNOLOGIA").isin("CABLE", "HYBRID FIBER COAXIAL (HFC)"), "CABLE/HFC")
    .when(col("TECNOLOGIA") == "XDSL", "XDSL")
    .when(col("TECNOLOGIA") == "SATELITAL", "SATELITAL")
    .when(col("TECNOLOGIA").isin("WIFI", "WIMAX", "OTRAS TECNOLOGÍAS INALÁMBRICAS"), "INALÁMBRICO")
    .otherwise("OTRAS")
)

print("Accesos por TIPO_TECNOLOGIA:")
df_internet.groupBy("TIPO_TECNOLOGIA") \
    .agg(Fsum("No DE ACCESOS").alias("total_accesos")) \
    .orderBy("total_accesos", ascending=False).show()

#4: Creacion columna
df_internet = df_internet.withColumn(
    "ES_RESIDENCIAL",
    when(col("SEGMENTO").startswith("RESIDENCIAL"), 1).otherwise(0)
)

print("Distribución ES_RESIDENCIAL:")
df_internet.groupBy("ES_RESIDENCIAL").agg(
    count("*").alias("registros"),
    Fsum("No DE ACCESOS").alias("total_accesos")
).show()

Columnas tras eliminación: 10
Accesos por TIPO_TECNOLOGIA:
+---------------+-------------+
|TIPO_TECNOLOGIA|total_accesos|
+---------------+-------------+
|      CABLE/HFC|      1344453|
|          FIBRA|      1216728|
|           XDSL|       708605|
|    INALÁMBRICO|       242319|
|      SATELITAL|        16711|
|          OTRAS|         8399|
+---------------+-------------+

Distribución ES_RESIDENCIAL:
+--------------+---------+-------------+
|ES_RESIDENCIAL|registros|total_accesos|
+--------------+---------+-------------+
|             0|    55871|       400072|
|             1|    85690|      3137143|
+--------------+---------+-------------+



In [0]:
print("--- REPORTE DE NULOS TRAS TRANSFORMACIONES ---")

expresiones_finales = []
for columna, tipo in df_internet.dtypes:
    if tipo == 'string':
        condicion = col(columna).isNull() | (col(columna) == "") | (col(columna) == "NaN")
    else:
        condicion = col(columna).isNull()
    expresiones_finales.append(count(when(condicion, columna)).alias(columna))

df_internet.select(expresiones_finales).show(vertical=True)

print("--- MUESTRA DEL DATASET TRANSFORMADO Y LIMPIO ---")
df_internet.select(
    "AÑO", "MUNICIPIO", "PROVEEDOR", "SEGMENTO",
    "TIPO_TECNOLOGIA", "VELOCIDAD_BAJADA", "VELOCIDAD_SUBIDA",
    "No DE ACCESOS", "ES_RESIDENCIAL"
).show(5, truncate=False)

print(f"\nFilas finales: {df_internet.count()}")
print(f"Columnas finales: {len(df_internet.columns)}")

--- REPORTE DE NULOS TRAS TRANSFORMACIONES ---
-RECORD 0---------------
 AÑO              | 0   
 TRIMESTRE        | 0   
 PROVEEDOR        | 0   
 COD_MUNICIPIO    | 0   
 MUNICIPIO        | 0   
 SEGMENTO         | 0   
 TECNOLOGIA       | 0   
 VELOCIDAD_BAJADA | 255 
 VELOCIDAD_SUBIDA | 261 
 No DE ACCESOS    | 0   
 TIPO_TECNOLOGIA  | 0   
 ES_RESIDENCIAL   | 0   

--- MUESTRA DEL DATASET TRANSFORMADO Y LIMPIO ---
+----+---------+-----------------------------------+-----------------------+---------------+----------------+----------------+-------------+--------------+
|AÑO |MUNICIPIO|PROVEEDOR                          |SEGMENTO               |TIPO_TECNOLOGIA|VELOCIDAD_BAJADA|VELOCIDAD_SUBIDA|No DE ACCESOS|ES_RESIDENCIAL|
+----+---------+-----------------------------------+-----------------------+---------------+----------------+----------------+-------------+--------------+
|2019|DUITAMA  |COMUNICACION CELULAR S A COMCEL S A|CORPORATIVO            |CABLE/HFC      |300.0           |